## This notebook tests Flash detection performance on the Kellect dataset.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import torch
from torch_geometric.data import Data
import os
import torch.nn.functional as F
import json
import warnings
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
warnings.filterwarnings('ignore')
from torch_geometric.loader import NeighborLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
%matplotlib inline

In [ ]:
from pprint import pprint
import gzip
from sklearn.manifold import TSNE
import json
import copy
import os

import gensim
from gensim.models import Word2Vec
from multiprocessing import Pool
from itertools import compress
from tqdm import tqdm
import time

import multiprocessing

import os
import json
import pickle

import torch
import numpy as np
from torch.nn import CrossEntropyLoss
from torch_geometric.data import Data
from torch_geometric.loader import NeighborLoader
from torch_geometric import utils
from sklearn.utils import class_weight

In [ ]:
def get_json_paths(root_dir):
    json_paths = []
    for dirpath, dirnames, filenames in os.walk(root_dir):
        for fname in filenames:
            if fname.lower().endswith('.json'):
                full_path = os.path.join(dirpath, fname)
                json_paths.append(full_path)
    return json_paths

root = "kellect/public_data"
paths = get_json_paths(root)
paths.sort()

In [ ]:
import json

def read_json_records(paths):
    records = []
    for path in paths:
        with open(path, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line:  # skip empty lines
                    records.append(json.loads(line))
    return records

In [ ]:
records = read_json_records(paths[30:60])
malicious_records = read_json_records(paths[:30])

In [ ]:
def construct_df(records):
    df = pd.DataFrame.from_records(records)
    events = ['FileIORead','FileIOWrite','ProcessStart','TcpIpRecvIPV4','TcpIpSendIPV4']
    df = df[df['Event'].isin(events)]
    df[['objectname', 'objectID', 'actorname', 'actorID', 'object']] = ''

    df.loc[df['Event']=='TcpIpSendIPV4', 'objectname'] = \
        df.loc[df['Event']=='TcpIpSendIPV4', 'args'].str['daddr']
    df.loc[df['Event']=='TcpIpRecvIPV4', 'objectname'] = \
        df.loc[df['Event']=='TcpIpRecvIPV4', 'args'].str['daddr']
    df.loc[df['Event']=='FileIORead',    'objectname'] = \
        df.loc[df['Event']=='FileIORead',    'args'].str['FileName']
    df.loc[df['Event']=='FileIOWrite',   'objectname'] = \
        df.loc[df['Event']=='FileIOWrite',   'args'].str['FileName']
    df.loc[df['Event']=='ProcessStart',  'objectname'] = \
        df.loc[df['Event']=='ProcessStart',  'args'].str['ImageFileName']
    
    df.loc[df['Event']=='TcpIpSendIPV4', 'objectID'] = \
        df.loc[df['Event']=='TcpIpSendIPV4', 'args'].str['daddr']
    df.loc[df['Event']=='TcpIpRecvIPV4', 'objectID'] = \
        df.loc[df['Event']=='TcpIpRecvIPV4', 'args'].str['daddr']
    df.loc[df['Event']=='FileIORead',    'objectID'] = \
        df.loc[df['Event']=='FileIORead',    'args'].str['FileName']
    df.loc[df['Event']=='FileIOWrite',   'objectID'] = \
        df.loc[df['Event']=='FileIOWrite',   'args'].str['FileName']
    df.loc[df['Event']=='ProcessStart',  'objectID'] = \
        df.loc[df['Event']=='ProcessStart',  'args'].str['ProcessId']
    
    df.loc[df['Event']=='TcpIpSendIPV4', 'actorID'] = \
        df.loc[df['Event']=='TcpIpSendIPV4', 'PID']
    df.loc[df['Event']=='TcpIpRecvIPV4', 'actorID'] = \
        df.loc[df['Event']=='TcpIpRecvIPV4', 'PID']
    df.loc[df['Event']=='FileIORead',    'actorID'] = \
        df.loc[df['Event']=='FileIORead',    'PID']
    df.loc[df['Event']=='FileIOWrite',   'actorID'] = \
        df.loc[df['Event']=='FileIOWrite',   'PID']
    df.loc[df['Event']=='ProcessStart',  'actorID'] = \
        df.loc[df['Event']=='ProcessStart',  'PPID']
    
    df.loc[df['Event']=='TcpIpSendIPV4', 'actorname'] = \
        df.loc[df['Event']=='TcpIpSendIPV4', 'PName']
    df.loc[df['Event']=='TcpIpRecvIPV4', 'actorname'] = \
        df.loc[df['Event']=='TcpIpRecvIPV4', 'PName']
    df.loc[df['Event']=='FileIORead',    'actorname'] = \
        df.loc[df['Event']=='FileIORead',    'PName']
    df.loc[df['Event']=='FileIOWrite',   'actorname'] = \
        df.loc[df['Event']=='FileIOWrite',   'PName']
    df.loc[df['Event']=='ProcessStart',  'actorname'] = \
        df.loc[df['Event']=='ProcessStart',  'PName']
    
    df.loc[df['Event']=='TcpIpSendIPV4', 'object'] = 'FLOW'
    df.loc[df['Event']=='TcpIpRecvIPV4', 'object'] = 'FLOW'
    df.loc[df['Event']=='FileIORead',    'object'] = 'FILE'
    df.loc[df['Event']=='FileIOWrite',   'object'] = 'FILE'
    df.loc[df['Event']=='ProcessStart',  'object'] = 'PROCESS'
    
    df = df[['Event','actorID','objectID','object','actorname','objectname','TimeStamp']]
    df = df.rename(columns={'Event': 'action'})
    
    return df

In [ ]:
def preprocess(df):
    df = df[df['actorID'] != df['objectID']]
    df = df.drop_duplicates(
        subset=['action','actorID','objectID','object','actorname','objectname'],
        keep='first'
    )
    return df

In [ ]:
def prepare_graph(df):
    nodes = {}
    labels = {}
    edges = []

    dummies = {'PROCESS': 0, 'FLOW': 1, 'FILE': 2}
    lblmap = {}

    for i in range(len(df)):
        x = df.iloc[i]
        actorid = x['actorID']
        objectid = x['objectID']

        if actorid not in nodes:
            nodes[actorid] = []
        if objectid not in nodes:
            nodes[objectid] = []

        nodes[actorid] += [x['actorname'], x['action'], x['objectname']]
        nodes[objectid] += [x['actorname'], x['action'], x['objectname']]

        labels[actorid] = dummies['PROCESS']
        labels[objectid] = dummies[x['object']]

        lblmap[actorid] = x['actorname']
        lblmap[objectid] = x['objectname']

        edges.append((actorid, objectid))

    features = []
    feat_labels = []
    edge_index = [[], []]
    index = {}
    mapp = []

    for k, v in nodes.items():
        features.append(v)
        feat_labels.append(labels[k])
        index[k] = len(features) - 1
        mapp.append(k)

    for src_id, dst_id in edges:
        src = index[src_id]
        dst = index[dst_id]
        edge_index[0].append(src)
        edge_index[1].append(dst)

    return features, feat_labels, edge_index, mapp, lblmap


In [ ]:
from gensim.models.callbacks import CallbackAny2Vec
from gensim.models import Word2Vec

class EpochSaver(CallbackAny2Vec):
    '''Callback to save model after each epoch.'''

    def __init__(self):
        self.epoch = 0

    def on_epoch_end(self, model):
        model.save('flash_kellect.model')
        self.epoch += 1

In [ ]:
class EpochLogger(CallbackAny2Vec):
    '''Callback to log information about training'''

    def __init__(self):
        self.epoch = 0

    def on_epoch_begin(self, model):
        print("Epoch #{} start".format(self.epoch))

    def on_epoch_end(self, model):
        print("Epoch #{} end".format(self.epoch))
        self.epoch += 1

In [ ]:
df = construct_df(records)
df = preprocess(df)
phrases, labels ,edges, mapp, lblmapp = prepare_graph(df)

In [ ]:
logger = EpochLogger()
saver = EpochSaver()
word2vec = Word2Vec(sentences=phrases, vector_size=20, window=5, min_count=1, workers=4,epochs=50,callbacks=[saver,logger])

In [ ]:
import torch
import math

class PositionalEncoder:

    def __init__(self, d_model, max_len=100000):
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        self.pe = torch.zeros(max_len, d_model)
        self.pe[:, 0::2] = torch.sin(position * div_term)
        self.pe[:, 1::2] = torch.cos(position * div_term)

    def embed(self, x):
        return x + self.pe[:x.size(0)]


def infer(document):
    word_embeddings = [w2vmodel.wv[word] for word in document if word in  w2vmodel.wv]
    
    if not word_embeddings:
        return np.zeros(30)

    output_embedding = torch.tensor(word_embeddings, dtype=torch.float)
    if len(document) < 100000:
        output_embedding = encoder.embed(output_embedding)

    output_embedding = output_embedding.detach().cpu().numpy()
    return np.mean(output_embedding, axis=0)

encoder = PositionalEncoder(20)
w2vmodel = Word2Vec.load("flash_kellect.model")

In [ ]:
from torch_geometric.nn import SAGEConv
import torch.nn.functional as F
import torch.nn as nn

class GCN(nn.Module):
    def __init__(self, in_channels=20, hidden_channels=32, num_classes=3):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels, normalize=True)
        self.conv2 = SAGEConv(hidden_channels, hidden_channels, normalize=True)
        self.lin   = nn.Linear(hidden_channels, num_classes)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)

        x = self.conv2(x, edge_index)
        x = F.relu(x)

        x = self.lin(x)
        return F.softmax(x, dim=1)

In [ ]:
import torch.nn.functional as F
from torch.nn import CrossEntropyLoss

In [ ]:
model = GCN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001, weight_decay=5e-4)

nodes = [infer(x) for x in phrases]
nodes = np.array(nodes)

criterion = CrossEntropyLoss()
graph = Data(
    x=torch.tensor(nodes, dtype=torch.float).to(device),
    y=torch.tensor(labels, dtype=torch.long).to(device),
    edge_index=torch.tensor(edges, dtype=torch.long).to(device)
)

for epochs in range(1000):
    model.train()
    optimizer.zero_grad() 
    out = model(graph.x, graph.edge_index) 
    loss = criterion(out, graph.y) 
    loss.backward() 
    optimizer.step()      
    total_loss = loss.item() 
    print(total_loss)

torch.save(model.state_dict(), 'flash_kellect.pth')

# Testing

# df = construct_df(malicious_records)
df = preprocess(df)
phrases, labels ,edges,mapp,lblmap = prepare_graph(df)

In [ ]:
model = GCN().to(device)
model.load_state_dict(torch.load('flash_kellect.pth'))

nodes = [infer(x) for x in phrases]
nodes = np.array(nodes)

criterion = CrossEntropyLoss()
graph = Data(
    x=torch.tensor(nodes, dtype=torch.float).to(device),
    y=torch.tensor(labels, dtype=torch.long).to(device),
    edge_index=torch.tensor(edges, dtype=torch.long).to(device)
)

In [ ]:
model.eval()
out = model(graph.x, graph.edge_index)

sorted, indices = out.sort(dim=1,descending=True)
conf = (sorted[:,0] - sorted[:,1]) / sorted[:,0]
conf = (conf - conf.min()) / conf.max()

pred = indices[:,0]
flag = (pred == graph.y) # & (conf >= 0.1)

index = utils.mask_to_index(~flag).tolist()

In [ ]:
ids = set([mapp[x] for x in index if labels[x] == 0])
names = set([lblmap[x] for x in ids])

In [ ]:
print(len(phrases))
print(len(names))

In [ ]:
yhat = set([
    "mimikatz.exe",
    "createdump.exe",
    "notepad++.exe",
    "powershell.exe",
    "nanodump.x64.exe",
    "Outflank-Dumpert.exe",
    'WmiPrvSE.exe',
    'WMIC.exe',
    'WmiApSrv.exe',
    'whoami.exe',
    'procdump.exe',
    'procdump64.exe',
    'xordump.exe',
    'notepad.exe',
    'cmd.exe'
])
y = names

In [ ]:
def compute_metrics(yhat, y):
    # Convert to sets
    yhat_set = set(yhat)
    y_set    = set(y)

    # Counts
    tp = len(yhat_set & y_set)       
    fp = len(yhat_set - y_set)       
    fn = len(y_set - yhat_set)       

    # Precision / Recall
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall    = tp / (tp + fn) if (tp + fn) else 0.0

    # F1-score
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

    return {
        'tp': tp,
        'fp': fp,
        'fn': fn,
        'tn': len(phrases)-tp-fp-fn,
        'precision': precision,
        'recall': recall,
        'f1': f1,}

In [ ]:
metrics = compute_metrics(y, yhat)